In [3]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT_STR = "/content/drive/MyDrive/Indic-Multimodal-NMT"

PROJECT_ROOT = Path(
    PROJECT_ROOT_STR
)

metadata_path = (
    PROJECT_ROOT_STR + "/data/processed/flickr30k_metadata.csv"
)


df = pd.read_csv(metadata_path)

df.head()

,image_id,filename,source_text
0,0,1000092795.jpg,Two young guys with shaggy hair look at their ...
1,0,1000092795.jpg,"Two young, White males are outside near many b..."
2,0,1000092795.jpg,Two men in green shirts are standing in a yard.
3,0,1000092795.jpg,A man in a blue shirt standing in a garden.
4,0,1000092795.jpg,Two friends enjoy time spent together.


In [ ]:
!pip install transformers sentencepiece sacremoses

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 13.2 MB/s eta 0:00:00


In [10]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "facebook/mbart-large-50-many-to-many-mmt"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

# Set source language
tokenizer.src_lang = "en_XX"

text = ["A boy is playing with a dog."]

inputs = tokenizer(text, return_tensors="pt", padding=True).to(device)

outputs = model.generate(
    **inputs,
    forced_bos_token_id=tokenizer.lang_code_to_id["hi_IN"],
    max_length=128,
)

translated = tokenizer.batch_decode(outputs, skip_special_tokens=True)

print(translated)

config.json:   0%|          | 0.00/1.43k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/529 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/649 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.44GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/261 [00:00<?, ?B/s]

['एक लड़का कुत्ते के साथ खेल रहा है।']


In [11]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model.to(device)

MBartForConditionalGeneration(
  (model): MBartModel(
    (shared): MBartScaledWordEmbedding(250054, 1024, padding_idx=1)
    (encoder): MBartEncoder(
      (embed_tokens): MBartScaledWordEmbedding(250054, 1024, padding_idx=1)
      (embed_positions): MBartLearnedPositionalEmbedding(1026, 1024)
      (layers): ModuleList(
        (0-11): 12 x MBartEncoderLayer(
          (self_attn): MBartAttention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (activation_fn): ReLU()
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=4096, out_features=1024, bias=True)
        

In [ ]:
text = [
    "A boy is playing with a dog."
]


inputs = tokenizer(
    text,
    return_tensors="pt",
    padding=True
).to(device)

outputs = model.generate(
    **inputs,
    forced_bos_token_id=tokenizer.lang_code_to_id["hi_IN"],
    max_length=128,
)

translated = tokenizer.batch_decode(
    outputs,
    skip_special_tokens=True
)


translated

['एक लड़का कुत्ते के साथ खेल रहा है।']

In [ ]:
print(df.head())
print(len(df))

   image_id        filename                                        source_text
0         0  1000092795.jpg  Two young guys with shaggy hair look at their ...
1         0  1000092795.jpg  Two young, White males are outside near many b...
2         0  1000092795.jpg    Two men in green shirts are standing in a yard.
3         0  1000092795.jpg        A man in a blue shirt standing in a garden.
4         0  1000092795.jpg             Two friends enjoy time spent together.
155070


In [8]:
def translate_batch(texts):

    inputs = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=128
    ).to(device)

    with torch.no_grad():

        generated = model.generate(
            **inputs,
            forced_bos_token_id=tokenizer.lang_code_to_id["hi_IN"],
            max_length=128
        )

    translations = tokenizer.batch_decode(
        generated,
        skip_special_tokens=True
    )

    return translations

In [ ]:
sample = df.head(10).copy()

sample["hindi"] = translate_batch(
    sample["source_text"].tolist()
)

sample[
    ["source_text", "hindi"]
]

,source_text,hindi
0,Two young guys with shaggy hair look at their ...,दो बूढ़े-बूढ़े बाल वाले लड़के yard में बैठे-बै...
1,"Two young, White males are outside near many b...",दो युवा सफेद नर बाहर अनेक वृक्षों के पास रहते ...
2,Two men in green shirts are standing in a yard.,हरे-भरे कमर पहने दो आदमी एक आँगन में खड़े हैं।
3,A man in a blue shirt standing in a garden.,एक नीली शरीर पहने हुए आदमी जो एक बाग में खड़ा है।
4,Two friends enjoy time spent together.,दो मित्रों को एक साथ बिताने का आनंद होता है।
5,Several men in hard hats are operating a giant...,भारी टोपी पहने हुए कई लोग एक विशाल पंखा प्रणाल...
6,Workers look down from up above on a piece of ...,मजदूर ऊपर से एक उपकरण पर नीचे देख रहे हैं।
7,Two men working on a machine wearing hard hats.,कठोर टोपी पहने हुए एक मशीन पर काम करने वाले दो...
8,Four men on top of a tall structure.,एक ऊंची इमारत के ऊपर चार आदमी।
9,Three men on a large rig.,एक बड़े यंत्र पर तीन आदमी।


In [5]:
from pathlib import Path
import pandas as pd
import torch
from tqdm import tqdm


CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"

CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


CHECKPOINT_FILE = CHECKPOINT_DIR / "translation_checkpoint.csv"

print(CHECKPOINT_FILE)

/content/drive/MyDrive/Indic-Multimodal-NMT/checkpoints/translation_checkpoint.csv


In [13]:
if CHECKPOINT_FILE.exists():

    translated_df = pd.read_csv(
        CHECKPOINT_FILE
    )

    start_index = len(translated_df)

    print(
        f"Resuming from {start_index} samples"
    )

else:

    translated_df = pd.DataFrame(
        columns=[
            "image_id",
            "filename",
            "source_text",
            "target_text"
        ]
    )

    start_index = 0

    print("Starting new translation job")

Resuming from 64 samples


In [ ]:
BATCH_SIZE = 32

SAVE_EVERY = 100   # batches

# tqdm is used here to display a smart progress bar, visualizing the progress of the batch translation.
# It wraps around any iterable (in this case, the range of indices) and provides real-time updates
# on the number of iterations completed, elapsed time, and estimated time remaining.
for batch_num, start in enumerate(
    tqdm(
        range(
            start_index,
            len(df),
            BATCH_SIZE
        )
    )
):

    batch_df = df.iloc[
        start:start+BATCH_SIZE
    ]


    translations = translate_batch(
        batch_df["source_text"].tolist()
    )


    batch_result = pd.DataFrame(
        {
            "image_id":
                batch_df["image_id"].values,

            "filename":
                batch_df["filename"].values,

            "source_text":
                batch_df["source_text"].values,

            "target_text":
                translations
        }
    )


    translated_df = pd.concat(
        [
            translated_df,
            batch_result
        ],
        ignore_index=True
    )


    # checkpoint save
    if batch_num % SAVE_EVERY == 0:

        translated_df.to_csv(
            CHECKPOINT_FILE,
            index=False
        )

        print(
            f"\nSaved checkpoint at {len(translated_df)} samples"
        )


  0%|          | 1/4844 [03:01<244:18:31, 181.60s/it]


Saved checkpoint at 96 samples


  1%|          | 33/4844 [2:06:09<274:04:41, 205.09s/it]